In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

In [1]:
np.random.seed(42)
dates = pd.date_range(start="2025-01-01", periods=180, freq="D")
categories = ["Electronics", "Clothing", "Home & Kitchen"]

data = {
    "Date": np.random.choice(dates, 500),
    "Category": np.random.choice(categories, 500),
    "Sales": np.random.normal(loc=200, scale=50, size=500).round(2),
    "Units_Sold": np.random.randint(1, 10, size=500)
}

df = pd.DataFrame(data)
df["Sales"] = df["Sales"].clip(lower=20) # Avoid negative values

category_dropdown = widgets.Dropdown(
    options=["All"] + categories,
    value="All",
    description="Category:",
    style={'description_width': 'initial'}
)


multiplier_slider = widgets.FloatSlider(
    value=1.0,
    min=0.5,
    max=2.0,
    step=0.1,
    description="Sales Multiplier:",
    continuous_update=False,
    style={'description_width': 'initial'}
)


chart_type_radio = widgets.RadioButtons(
    options=["Line Chart (Over Time)", "Bar Chart (Total Sales)", "Scatter (Sales vs Units)"],
    value="Line Chart (Over Time)",
    description="Chart Type:",
    style={'description_width': 'initial'}
)

date_picker_start = widgets.DatePicker(
    description='Start Date',
    value=df['Date'].min().date()
)
output = widgets.Output()
def update_dashboard(change=None):
    with output:
        clear_output(wait=True)
        
        # Filter Data
        filtered_df = df.copy()
        
        if category_dropdown.value != "All":
            filtered_df = filtered_df[filtered_df["Category"] == category_dropdown.value]
            
        if date_picker_start.value:
            filtered_df = filtered_df[filtered_df["Date"].dt.date >= date_picker_start.value]

        # Apply Multiplier
        filtered_df["Adjusted_Sales"] = filtered_df["Sales"] * multiplier_slider.value

        
        total_sales = filtered_df["Adjusted_Sales"].sum()
        total_units = filtered_df["Units_Sold"].sum()
        avg_sale = filtered_df["Adjusted_Sales"].mean()
        
        print(f"📊 SUMMARY METRICS")
        print(f"Total Revenue: ${total_sales:,.2f} | Total Units Sold: {total_units:,} | Avg Transaction: ${avg_sale:.2f}")
        print("-" * 75)


        fig, ax = plt.subplots(figsize=(10, 4))
        
        if chart_type_radio.value == "Line Chart (Over Time)":
            daily_sales = filtered_df.groupby("Date")["Adjusted_Sales"].sum().reset_index()
            ax.plot(daily_sales["Date"], daily_sales["Adjusted_Sales"], color="#2b5c8f", linewidth=2)
            ax.set_title("Adjusted Daily Sales Over Time")
            ax.set_ylabel("Sales ($)")
            
        elif chart_type_radio.value == "Bar Chart (Total Sales)":
            cat_sales = filtered_df.groupby("Category")["Adjusted_Sales"].sum()
            cat_sales.plot(kind="bar", ax=ax, color="#4caf50")
            ax.set_title("Total Adjusted Sales by Category")
            ax.set_ylabel("Sales ($)")
            plt.xticks(rotation=0)
            
        elif chart_type_radio.value == "Scatter (Sales vs Units)":
            ax.scatter(filtered_df["Units_Sold"], filtered_df["Adjusted_Sales"], alpha=0.6, color="#ff9800")
            ax.set_title("Sales ($) vs. Units Sold")
            ax.set_xlabel("Units Sold")
            ax.set_ylabel("Adjusted Sales ($)")

        plt.grid(True, linestyle="--", alpha=0.5)
        plt.tight_layout()
        plt.show()
category_dropdown.observe(update_dashboard, names='value')
multiplier_slider.observe(update_dashboard, names='value')
chart_type_radio.observe(update_dashboard, names='value')
date_picker_start.observe(update_dashboard, names='value')

# Layout arrangement using VBox and HBox
controls = widgets.VBox([
    widgets.HBox([category_dropdown, date_picker_start]),
    widgets.HBox([multiplier_slider, chart_type_radio])
])

app_layout = widgets.VBox([
    widgets.HTML("<h2>📈 Interactive Sales Dashboard</h2>"),
    controls,
    widgets.HTML("<hr>"),
    output
])
display(app_layout)
update_dashboard()

NameError: name 'np' is not defined